# Spam Classification with XGBoost

This notebook builds a spam detection model using **XGBoost (Extreme Gradient Boosting)**, a powerful and efficient gradient boosting framework.

**Pipeline:**
1. Load and explore the SMS Spam dataset
2. Preprocess text with TF-IDF vectorization
3. Train an XGBoost classifier (handling class imbalance)
4. Evaluate with Classification Report, Accuracy, and Confusion Matrix
5. Visualize feature importances

**Dataset:** SMS Spam Collection — labeled `ham` (not spam) or `spam`.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## 2. Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('spam.csv', encoding='latin1')

# Keep only the relevant columns and rename them
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'message']

print('Dataset shape:', df.shape)
df.head()

In [ ]:
# Class distribution
print('Class distribution:')
print(df['label'].value_counts())
print(f"\nSpam ratio: {df['label'].value_counts(normalize=True)['spam']*100:.1f}%")

# Visualize class balance
fig, ax = plt.subplots(figsize=(5, 4))
df['label'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'], ax=ax, edgecolor='white')
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Label', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_xticklabels(['ham', 'spam'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2, p.get_height() + 20),
                ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

We convert text messages into numerical features using **TF-IDF (Term Frequency-Inverse Document Frequency)** vectorization.

- `max_features=5000` — keeps the top 5000 most informative words
- Labels are encoded as binary values: `ham = 0`, `spam = 1`
- `stratify=y` ensures the class ratio is preserved in both train and test sets

In [ ]:
# Encode labels: ham=0, spam=1
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['message'])
y = df['label_encoded']

print('Feature matrix shape:', X.shape)
print('Label distribution:')
print(y.value_counts().rename({0: 'ham (0)', 1: 'spam (1)'}))

In [ ]:
# Train/Test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')

# Class imbalance weight for XGBoost
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f'\nscale_pos_weight (ham/spam ratio): {scale_pos_weight:.2f}')

## 4. Train the XGBoost Classifier

**Key hyperparameters:**
- `scale_pos_weight` — compensates for class imbalance by upweighting the minority class (spam)
- `eval_metric='logloss'` — uses log loss as the evaluation metric during training
- `random_state=42` — ensures reproducibility

XGBoost builds trees sequentially, each correcting the errors of the previous one, which makes it highly effective for imbalanced text classification.

In [ ]:
# Train XGBoost
xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print('XGBoost model trained successfully.')

## 5. Model Evaluation

In [ ]:
# Classification Report
print('=== XGBoost Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

acc = accuracy_score(y_test, y_pred)
print(f'Overall Accuracy: {acc * 100:.2f}%')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['ham', 'spam'],
    yticklabels=['ham', 'spam'],
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (ham correctly identified) : {tn}')
print(f'False Positives (ham misclassified as spam): {fp}')
print(f'False Negatives (spam missed)              : {fn}')
print(f'True Positives  (spam correctly caught)    : {tp}')

## 6. Performance Metrics Summary

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True)

metrics = {
    'Accuracy':         round(acc * 100, 2),
    'Precision (spam)': round(report['1']['precision'] * 100, 2),
    'Recall (spam)':    round(report['1']['recall'] * 100, 2),
    'F1-Score (spam)':  round(report['1']['f1-score'] * 100, 2),
}

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metrics.keys(), metrics.values(), color='#DD8452', alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, metrics.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f'{val:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
ax.set_ylim(50, 108)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('XGBoost — Spam Classification Performance', fontsize=13, fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

print('\nMetrics Summary:')
for k, v in metrics.items():
    print(f'  {k:<22}: {v:.2f}%')

## 7. Top Feature Importances

XGBoost tracks which TF-IDF features (words) contributed most to the model's predictions. The chart below shows the **top 20 most important words** for spam detection.

In [ ]:
# Get feature importances
feature_names = tfidf.get_feature_names_out()
importances = xgb.feature_importances_

top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]
top_features = feature_names[top_idx]
top_scores = importances[top_idx]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_features[::-1], top_scores[::-1], color='#4C72B0', alpha=0.85, edgecolor='white')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title(f'Top {top_n} Most Important Words — XGBoost', fontsize=13, fontweight='bold')
ax.xaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 8. Conclusion

The **XGBoost classifier** was trained on TF-IDF features from the SMS Spam Collection dataset.

**Key takeaways:**
- Class imbalance (~87% ham / ~13% spam) was handled using `scale_pos_weight`
- The model achieves **high accuracy and strong F1-Score on the spam class**, making it well-suited for production spam filtering
- Feature importances reveal that words like *free*, *txt*, *call*, *claim*, and *prize* are the strongest spam indicators — consistent with real-world spam patterns

The trained model and TF-IDF vectorizer can be serialized (e.g., with `joblib`) and deployed in a spam detection pipeline.